In [1]:
import os
from pyspark.sql import SparkSession, functions as F

jar_path = f"{os.getcwd()}/postgresql-42.7.0.jar"

spark = (SparkSession.builder.appName("Gold-Layer")
    .config("spark.jars", jar_path)
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

SILVER = "/home/jovyan/work/data/silver"
GOLD = "/home/jovyan/work/data/gold"

fact = spark.read.format("delta").load(f"{SILVER}/fact_order_items")
dim_products = spark.read.format("delta").load(f"{SILVER}/dim_products")
dim_users = spark.read.format("delta").load(f"{SILVER}/dim_users")

# Filter orphans for Gold (business marts shouldn't have NULL dept)
fact_clean = fact.filter(F.col("department").isNotNull())

print(f"Silver fact rows: {fact.count():,}")
print(f"Clean fact (Gold input): {fact_clean.count():,}")

Silver fact rows: 32,434,489
Clean fact (Gold input): 32,434,486


In [2]:
#Mart 1: Department KPI

gold_dept_kpi = (fact_clean
    .groupBy("department")
    .agg(
        F.count("*").alias("total_items_sold"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.sum("reordered").alias("reorder_count"),
        F.round(F.avg("reordered"), 4).alias("reorder_rate"),
        F.round(F.count("*") / F.countDistinct("order_id"), 2).alias("avg_items_per_order"),
        F.current_timestamp().alias("gold_loaded_at")
    )
    .orderBy(F.desc("reorder_rate")))

(gold_dept_kpi.write
    .format("delta").mode("overwrite")
    .save(f"{GOLD}/dept_kpi"))

print("✅ gold_dept_kpi written")
gold_dept_kpi.show(25, truncate=False)

✅ gold_dept_kpi written
+---------------+----------------+-------------+----------------+-------------+------------+-------------------+-------------------------+
|department     |total_items_sold|unique_orders|unique_customers|reorder_count|reorder_rate|avg_items_per_order|gold_loaded_at           |
+---------------+----------------+-------------+----------------+-------------+------------+-------------------+-------------------------+
|dairy eggs     |5414016         |2177338      |190565          |3627221      |0.67        |2.49               |2026-05-02 18:06:09.02457|
|beverages      |2690129         |1457351      |172795          |1757892      |0.6535      |1.85               |2026-05-02 18:06:09.02457|
|produce        |9479291         |2409320      |193237          |6160710      |0.6499      |3.93               |2026-05-02 18:06:09.02457|
|bakery         |1176787         |881556       |140612          |739188       |0.6281      |1.33               |2026-05-02 18:06:09.02457|
|de

In [3]:
#Mart 2: SKU Performance
gold_sku_performance = (fact_clean
    .groupBy("product_id")
    .agg(
        F.count("*").alias("total_sold"),
        F.countDistinct("user_id").alias("unique_buyers"),
        F.round(F.avg("reordered"), 4).alias("reorder_rate"),
        F.round(F.avg("add_to_cart_order"), 2).alias("avg_cart_position")
    )
    .join(F.broadcast(dim_products.select("product_id", "product_name", "department", "aisle")),
          on="product_id", how="inner")
    .select(
        "product_id", "product_name", "department", "aisle",
        "total_sold", "unique_buyers", "reorder_rate", "avg_cart_position",
        F.current_timestamp().alias("gold_loaded_at")
    )
    .orderBy(F.desc("total_sold")))

(gold_sku_performance.write
    .format("delta").mode("overwrite")
    .save(f"{GOLD}/sku_performance"))

print(f"✅ gold_sku_performance: {gold_sku_performance.count():,} rows")
gold_sku_performance.show(15, truncate=False)

✅ gold_sku_performance: 49,676 rows
+----------+----------------------+----------+--------------------------+----------+-------------+------------+-----------------+--------------------------+
|product_id|product_name          |department|aisle                     |total_sold|unique_buyers|reorder_rate|avg_cart_position|gold_loaded_at            |
+----------+----------------------+----------+--------------------------+----------+-------------+------------+-----------------+--------------------------+
|24852     |Banana                |produce   |fresh fruits              |472565    |73956        |0.8435      |4.89             |2026-05-02 18:06:26.910242|
|13176     |Bag of Organic Bananas|produce   |fresh fruits              |379450    |63537        |0.8326      |5.1              |2026-05-02 18:06:26.910242|
|21137     |Organic Strawberries  |produce   |fresh fruits              |264683    |58838        |0.7777      |7.25             |2026-05-02 18:06:26.910242|
|21903     |Organic Ba

In [4]:
#Mart 3: Hourly Demand

gold_hourly_demand = (fact_clean
    .groupBy("order_dow", "order_hour_of_day", "department")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.count("*").alias("items"),
        F.current_timestamp().alias("gold_loaded_at")
    ))

(gold_hourly_demand.write
    .format("delta").mode("overwrite")
    .partitionBy("order_dow")
    .save(f"{GOLD}/hourly_demand"))

print(f"✅ gold_hourly_demand: {gold_hourly_demand.count():,} rows")

# Quick insight
peak = (gold_hourly_demand
    .groupBy("order_dow", "order_hour_of_day")
    .agg(F.sum("orders").alias("total_orders"))
    .orderBy(F.desc("total_orders"))
    .limit(10))
print("\n=== Top 10 Peak (DOW, Hour) ===")
peak.show()

✅ gold_hourly_demand: 3,528 rows

=== Top 10 Peak (DOW, Hour) ===
+---------+-----------------+------------+
|order_dow|order_hour_of_day|total_orders|
+---------+-----------------+------------+
|        0|               14|      262222|
|        0|               13|      259431|
|        0|               15|      258515|
|        0|               12|      247815|
|        0|               11|      246420|
|        1|               10|      242301|
|        0|               16|      235907|
|        0|               10|      235192|
|        1|               11|      228812|
|        1|                9|      222095|
+---------+-----------------+------------+



In [5]:
# Mart 4: Customer Segmentation

gold_customer_segment = (dim_users
    .withColumn("segment",
        F.when(F.col("total_orders") >= 50, "VIP")
         .when(F.col("total_orders") >= 20, "Loyal")
         .when(F.col("total_orders") >= 5, "Regular")
         .otherwise("New"))
    .withColumn("frequency_band",
        F.when(F.col("avg_days_between_orders") <= 7, "Weekly")
         .when(F.col("avg_days_between_orders") <= 14, "Bi-weekly")
         .when(F.col("avg_days_between_orders") <= 30, "Monthly")
         .otherwise("Occasional"))
    .select("user_id", "total_orders", "avg_days_between_orders",
            "segment", "frequency_band",
            F.current_timestamp().alias("gold_loaded_at")))

(gold_customer_segment.write
    .format("delta").mode("overwrite")
    .partitionBy("segment")
    .save(f"{GOLD}/customer_segment"))

print("=== Customer Segment Distribution ===")
gold_customer_segment.groupBy("segment").count().orderBy(F.desc("count")).show()

print("=== Frequency Band Distribution ===")
gold_customer_segment.groupBy("frequency_band").count().orderBy(F.desc("count")).show()


=== Customer Segment Distribution ===
+-------+------+
|segment| count|
+-------+------+
|Regular|128292|
|  Loyal| 42467|
|    New| 23986|
|    VIP| 11464|
+-------+------+

=== Frequency Band Distribution ===
+--------------+------+
|frequency_band| count|
+--------------+------+
|       Monthly|112159|
|     Bi-weekly| 69579|
|        Weekly| 24471|
+--------------+------+



In [6]:
# Push to PostgreSQL

PG_OPTS = {
    "url": "jdbc:postgresql://pg-warehouse:5432/warehouse",
    "user": "dataeng",
    "password": "dataeng",
    "driver": "org.postgresql.Driver"
}

top_sku = gold_sku_performance.orderBy(F.desc("total_sold")).limit(1000)

segment_summary = (gold_customer_segment
    .groupBy("segment", "frequency_band")
    .agg(
        F.count("*").alias("user_count"),
        F.round(F.avg("total_orders"), 1).alias("avg_orders"),
        F.round(F.avg("avg_days_between_orders"), 1).alias("avg_days")
    ))

tables_to_push = [
    ("kpi_dept", gold_dept_kpi),
    ("kpi_sku_top1000", top_sku),
    ("kpi_hourly_demand", gold_hourly_demand),
    ("kpi_customer_segment", segment_summary),
]

for table_name, df in tables_to_push:
    (df.write.format("jdbc")
        .options(**PG_OPTS, dbtable=table_name)
        .mode("overwrite").save())
    print(f"✅ Pushed {table_name} ({df.count():,} rows)")

print("\n🎉 All Gold marts pushed to Postgres serving layer")

✅ Pushed kpi_dept (21 rows)
✅ Pushed kpi_sku_top1000 (1,000 rows)
✅ Pushed kpi_hourly_demand (3,528 rows)
✅ Pushed kpi_customer_segment (11 rows)

🎉 All Gold marts pushed to Postgres serving layer


In [7]:
# Read back from Postgres
for table in ["kpi_dept", "kpi_sku_top1000", "kpi_hourly_demand", "kpi_customer_segment"]:
    df = (spark.read.format("jdbc")
        .options(**PG_OPTS, dbtable=table)
        .load())
    print(f"\n=== {table} ({df.count():,} rows) ===")
    df.show(5, truncate=False)


=== kpi_dept (21 rows) ===
+----------+----------------+-------------+----------------+-------------+------------+-------------------+--------------------------+
|department|total_items_sold|unique_orders|unique_customers|reorder_count|reorder_rate|avg_items_per_order|gold_loaded_at            |
+----------+----------------+-------------+----------------+-------------+------------+-------------------+--------------------------+
|dairy eggs|5414016         |2177338      |190565          |3627221      |0.67        |2.49               |2026-05-02 18:06:42.879685|
|beverages |2690129         |1457351      |172795          |1757892      |0.6535      |1.85               |2026-05-02 18:06:42.879685|
|produce   |9479291         |2409320      |193237          |6160710      |0.6499      |3.93               |2026-05-02 18:06:42.879685|
|bakery    |1176787         |881556       |140612          |739188       |0.6281      |1.33               |2026-05-02 18:06:42.879685|
|deli      |1051249        